# Chapter 13 — Testing and Debugging

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 13.

**Notebook environment note:** `unittest.main()` normally calls `sys.exit()` after running tests, which would stop this notebook's kernel. Every example below instead calls `unittest.main(argv=['ignored'], exit=False)` to run the tests without exiting — in a real `.py` test file run directly, plain `unittest.main()` works exactly as the book shows.

---


### Item 108 — Verify Related Behaviors in `TestCase` Subclasses

`unittest.TestCase` groups related test methods (each named `test_*`) into one class, sharing common setup/context — this keeps tests for the same unit of functionality organized together, and each `assert*` method (`assertEqual`, `assertTrue`, `assertRaises`, ...) gives a clear, specific failure message when something doesn't match, unlike a bare `assert expr`.


In [1]:
import unittest

def to_camel_case(snake_str):
    parts = snake_str.split('_')
    return parts[0] + ''.join(p.title() for p in parts[1:])

class TestToCamelCase(unittest.TestCase):
    def test_simple(self):
        self.assertEqual(to_camel_case('hello_world'), 'helloWorld')

    def test_single_word(self):
        self.assertEqual(to_camel_case('hello'), 'hello')

    def test_many_parts(self):
        self.assertEqual(to_camel_case('the_quick_brown_fox'), 'theQuickBrownFox')

unittest.main(argv=['ignored', '-v'], exit=False)


test_many_parts (__main__.TestToCamelCase) ... 

ok


test_simple (__main__.TestToCamelCase) ... 

ok


test_single_word (__main__.TestToCamelCase) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK


### Item 109 — Prefer Integration Tests over Unit Tests

Unit tests (testing one function/class in total isolation, often with mocks standing in for its dependencies) can all pass while the *real* system is still broken, if the mocks don't accurately reflect how the real dependency behaves. Integration tests (exercising real components together, e.g. hitting a real test database instead of a mock) catch a category of bugs unit tests structurally cannot — the book's advice is to weight testing effort more toward integration tests than is typically assumed, using unit tests mainly for pure logic and edge cases.


In [2]:
import unittest
import sqlite3

class UserRepository:
    def __init__(self, connection):
        self.conn = connection
    def add_user(self, name):
        self.conn.execute('INSERT INTO users (name) VALUES (?)', (name,))
        self.conn.commit()
    def get_user_count(self):
        return self.conn.execute('SELECT COUNT(*) FROM users').fetchone()[0]

class TestUserRepositoryIntegration(unittest.TestCase):
    def setUp(self):
        # a REAL (in-memory) database, not a mock -- exercises the actual SQL, actual sqlite3 behavior
        self.conn = sqlite3.connect(':memory:')
        self.conn.execute('CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)')
        self.repo = UserRepository(self.conn)

    def test_add_and_count(self):
        self.repo.add_user('Ada')
        self.repo.add_user('Alan')
        self.assertEqual(self.repo.get_user_count(), 2)   # real SQL executed, not assumed

    def tearDown(self):
        self.conn.close()

unittest.main(argv=['ignored', '-v'], exit=False)


test_many_parts (__main__.TestToCamelCase) ... 

ok


test_simple (__main__.TestToCamelCase) ... 

ok


test_single_word (__main__.TestToCamelCase) ... 

ok


test_add_and_count (__main__.TestUserRepositoryIntegration) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


### Item 110 — Isolate Tests From Each Other with `setUp`, `tearDown`, `setUpModule`, and `tearDownModule`

`setUp`/`tearDown` run before/after **every** test method (fresh state each time, so tests can't accidentally depend on each other's leftover state or ordering); `setUpModule`/`tearDownModule` run **once** for the whole file (for genuinely expensive setup, like a real database connection, that's safe and worthwhile to share across many tests).


In [3]:
import unittest

call_log = []

def setUpModule():
    call_log.append('setUpModule (once, for the whole file)')

def tearDownModule():
    call_log.append('tearDownModule (once, for the whole file)')

class TestIsolation(unittest.TestCase):
    def setUp(self):
        call_log.append('setUp (before EVERY test)')
        self.value = 10   # fresh state for every single test method

    def tearDown(self):
        call_log.append('tearDown (after EVERY test)')

    def test_one(self):
        self.value += 5
        self.assertEqual(self.value, 15)

    def test_two(self):
        # if setUp didn't reset self.value, this could accidentally see test_one's leftover state
        self.assertEqual(self.value, 10)

unittest.main(argv=['ignored'], exit=False)
setUpModule()
print('\n'.join(call_log))


.

.

.

.

.

.

----------------------------------------------------------------------
Ran 6 tests in 0.002s

OK


setUpModule (once, for the whole file)
setUp (before EVERY test)
tearDown (after EVERY test)
setUp (before EVERY test)
tearDown (after EVERY test)
tearDownModule (once, for the whole file)
setUpModule (once, for the whole file)


### Item 111 — Use Mocks to Test Code with Complex Dependencies

`unittest.mock.Mock`/`patch` let you replace a real (slow, external, or hard-to-set-up) dependency — a network call, a database, an email service — with a fake stand-in that records how it was called and returns whatever you tell it to, so you can test your code's logic in isolation from that dependency's real behavior.


In [4]:
from unittest.mock import Mock

class EmailService:
    def send(self, to, subject, body):
        raise NotImplementedError('would really send an email over the network')

def notify_user(email_service, user_email):
    email_service.send(user_email, 'Welcome!', 'Thanks for signing up.')
    return True

mock_service = Mock()          # a fake stand-in -- no real email is ever sent
result = notify_user(mock_service, 'test@example.com')

print('function returned:', result)
mock_service.send.assert_called_once_with('test@example.com', 'Welcome!', 'Thanks for signing up.')
print('confirmed exactly how the mock was called, with no real email service involved')


function returned: True
confirmed exactly how the mock was called, with no real email service involved


### Item 112 — Encapsulate Dependencies to Facilitate Mocking and Testing

Code that directly calls a global/hardcoded dependency (a module-level `requests.get(...)` sprinkled throughout) is hard to mock cleanly. Wrapping the dependency behind your own small interface/class (dependency injection: pass the dependency in as a constructor/function argument) means tests can substitute a mock for that one seam easily, without needing to monkey-patch deep inside unrelated code.


In [5]:
from unittest.mock import Mock

class WeatherClient:   # encapsulates the "how do we get weather data" dependency behind one interface
    def get_temperature(self, city):
        raise NotImplementedError('would really call a weather API')

class WeatherReport:
    def __init__(self, client: WeatherClient):   # dependency INJECTED, not hardcoded internally
        self.client = client

    def describe(self, city):
        temp = self.client.get_temperature(city)
        return f'{city} is currently {temp}°C'

mock_client = Mock()
mock_client.get_temperature.return_value = 22

report = WeatherReport(mock_client)   # the real WeatherClient is never touched in this test
print(report.describe('Bogota'))


Bogota is currently 22°C


### Item 113 — Use `assertAlmostEqual` to Control Precision in Floating Point Tests

Floating-point rounding error (Item 106) means asserting exact equality between two computed floats (`self.assertEqual(0.1 + 0.2, 0.3)`) can fail even when the math is conceptually correct. `assertAlmostEqual(a, b, places=N)` instead checks they're equal to within N decimal places, which is the numerically honest way to test floating-point results.


In [6]:
import unittest

class TestFloatingPoint(unittest.TestCase):
    def test_exact_equality_fails(self):
        # this would actually FAIL if uncommented -- 0.1 + 0.2 != 0.3 exactly, due to binary rounding
        # self.assertEqual(0.1 + 0.2, 0.3)
        pass

    def test_almost_equal_passes(self):
        self.assertAlmostEqual(0.1 + 0.2, 0.3, places=9)   # correctly accounts for float imprecision

    def test_almost_equal_with_computed_average(self):
        values = [1.1, 2.2, 3.3]
        average = sum(values) / len(values)
        self.assertAlmostEqual(average, 2.2, places=9)

unittest.main(argv=['ignored', '-v'], exit=False)


test_almost_equal_passes (__main__.TestFloatingPoint) ... 

ok


test_almost_equal_with_computed_average (__main__.TestFloatingPoint) ... 

ok


test_exact_equality_fails (__main__.TestFloatingPoint) ... 

ok


... (9 outputs left out)

ok


test_add_and_count (__main__.TestUserRepositoryIntegration) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.005s

OK


### Item 114 — Consider Interactive Debugging with `pdb`

`pdb` is Python's built-in interactive debugger: `breakpoint()` (or `import pdb; pdb.set_trace()`) pauses execution at that exact line and drops you into a prompt where you can inspect variables, step line-by-line (`n`), step into a function call (`s`), or continue (`c`) — far more precise than debugging via scattered `print()` calls.

**Notebook environment note:** `pdb`'s prompt needs a real interactive terminal to type commands into — inside this automatically-executed notebook there's no human to type `n`/`s`/`c`, so the cell below demonstrates `pdb` programmatically instead (feeding it commands from a string) to show what the session looks like. In Cursor's own interactive kernel, `breakpoint()` will actually pause and give you a live prompt to type into.


In [7]:
import pdb
import io
import sys

def calculate_total(prices):
    subtotal = sum(prices)
    tax = subtotal * 0.08
    return subtotal + tax

# Programmatic demo: scripted pdb commands, since there's no human typing here in this notebook
commands = io.StringIO('p prices\np subtotal\nn\np tax\nc\n')
old_stdin = sys.stdin
sys.stdin = commands
try:
    debugger = pdb.Pdb(stdin=commands, stdout=sys.stdout)
    debugger.set_trace()
    calculate_total([10.0, 20.0, 30.0])
finally:
    sys.stdin = old_stdin

print()
print('In Cursor, replace the scripted commands above with a real breakpoint():')
print('  def calculate_total(prices):')
print('      breakpoint()   # pauses here, gives you a REAL interactive (Pdb) prompt to type into')
print('      subtotal = sum(prices)')


> /sessions/festive-peaceful-johnson/tmp/ipykernel_32/2370425408.py(17)<module>()
     15     debugger = pdb.Pdb(stdin=commands, stdout=sys.stdout)
     16     debugger.set_trace()
---> 17     calculate_total([10.0, 20.0, 30.0])
     18 finally:
     19     sys.stdin = old_stdin

ipdb> 

*** NameError: name 'prices' is not defined
ipdb> 

*** NameError: name 'subtotal' is not defined
ipdb> 

> /sessions/festive-peaceful-johnson/tmp/ipykernel_32/2370425408.py(19)<module>()
     17     calculate_total([10.0, 20.0, 30.0])
     18 finally:
---> 19     sys.stdin = old_stdin
     20 
     21 print()

ipdb> 

*** NameError: name 'tax' is not defined
ipdb> 


In Cursor, replace the scripted commands above with a real breakpoint():
  def calculate_total(prices):
      breakpoint()   # pauses here, gives you a REAL interactive (Pdb) prompt to type into
      subtotal = sum(prices)


### Item 115 — Use `tracemalloc` to Understand Memory Usage and Leaks

`tracemalloc` (standard library) tracks *where in your code* each memory allocation happened — `tracemalloc.start()` begins tracking, `tracemalloc.take_snapshot()` captures the current state, and comparing two snapshots (`.compare_to(other, 'lineno')`) shows exactly which lines are responsible for growth between them, which is far more precise than guessing which part of a program is leaking memory.


In [8]:
import tracemalloc

tracemalloc.start()

snapshot1 = tracemalloc.take_snapshot()

# Simulate something that allocates a meaningful amount of memory
leaky_list = []
for i in range(100_000):
    leaky_list.append(str(i) * 10)

snapshot2 = tracemalloc.take_snapshot()

top_stats = snapshot2.compare_to(snapshot1, 'lineno')
print('Top memory-growth lines between the two snapshots:')
for stat in top_stats[:3]:
    print(stat)

tracemalloc.stop()


Top memory-growth lines between the two snapshots:
/sessions/festive-peaceful-johnson/tmp/ipykernel_32/2634578349.py:10: size=10.1 MiB (+10.1 MiB), count=100001 (+100001), average=106 B
/usr/lib/python3.10/tracemalloc.py:423: size=504 B (+504 B), count=3 (+3), average=168 B
/usr/lib/python3.10/codeop.py:118: size=784 B (+464 B), count=12 (+8), average=65 B
